In [0]:
dbutils.widgets.text("catalog_name", "adbsflordeoro")
dbutils.widgets.text("silver_schema", "retail_silver")

catalog_name = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")

print(f"Catalog: {catalog_name}")
print(f"Silver schema: {silver_schema}")

In [0]:
from pyspark.sql import functions as F

CATALOG = "adbsflordeoro"
BRONZE_SCHEMA = "default"
SILVER_SCHEMA = "retail_silver"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SILVER_SCHEMA}`"
)

print(f"Origen: {CATALOG}.{BRONZE_SCHEMA}")
print(f"Destino: {CATALOG}.{SILVER_SCHEMA}")

In [0]:

# Leer las nuevas tablas externas de Bronze

bronze_sales = spark.table(
    f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`bronze_train`"
)

bronze_transactions = spark.table(
    f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`bronze_transactions_ext`"
)

bronze_stores = spark.table(
    f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`bronze_stores_ext`"
)

# Normalizar los nombres de columnas
def normalize_columns(df):
    for old_name in df.columns:
        new_name = old_name.strip().lower()

        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

    return df

bronze_sales = normalize_columns(bronze_sales)
bronze_transactions = normalize_columns(bronze_transactions)
bronze_stores = normalize_columns(bronze_stores)

# Validar columnas requeridas
required_columns = {
    "sales": {"date", "store_nbr", "family", "sales", "onpromotion"},
    "transactions": {"date", "store_nbr", "transactions"},
    "stores": {"store_nbr", "city", "state", "type", "cluster"}
}

dataframes = {
    "sales": bronze_sales,
    "transactions": bronze_transactions,
    "stores": bronze_stores
}

for name, df in dataframes.items():
    missing = required_columns[name] - set(df.columns)

    if missing:
        raise ValueError(
            f"Faltan columnas en Bronze {name}: {sorted(missing)}"
        )

    print(f"{name}: columnas validadas; filas = {df.count()}")


In [0]:
silver_sales = (
    bronze_sales
    .select(
        F.col("id").cast("long").alias("sales_id")
        if "id" in bronze_sales.columns
        else F.lit(None).cast("long").alias("sales_id"),

        F.to_date(F.col("date")).alias("date"),
        F.col("store_nbr").cast("int").alias("store_nbr"),
        F.trim(F.col("family")).alias("family"),
        F.col("sales").cast("double").alias("sales"),
        F.col("onpromotion").cast("int").alias("onpromotion")
    )
)

# Reportar problemas de calidad antes de decidir cómo tratarlos
sales_quality = silver_sales.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.col("date").isNull().cast("int")).alias("null_dates"),
    F.sum(F.col("store_nbr").isNull().cast("int")).alias("null_store_nbr"),
    F.sum(F.col("family").isNull().cast("int")).alias("null_family"),
    F.sum(F.col("sales").isNull().cast("int")).alias("null_sales")
)

display(sales_quality)

In [0]:
silver_transactions = (
    bronze_transactions
    .select(
        F.to_date(F.col("date")).alias("date"),
        F.col("store_nbr").cast("int").alias("store_nbr"),
        F.col("transactions").cast("long").alias("transactions")
    )
)

transactions_quality = silver_transactions.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.col("date").isNull().cast("int")).alias("null_dates"),
    F.sum(F.col("store_nbr").isNull().cast("int")).alias("null_store_nbr"),
    F.sum(F.col("transactions").isNull().cast("int")).alias("null_transactions")
)

display(transactions_quality)

In [0]:
silver_stores = (
    bronze_stores
    .select(
        F.col("store_nbr").cast("int").alias("store_nbr"),
        F.trim(F.col("city")).alias("city"),
        F.trim(F.col("state")).alias("state"),
        F.trim(F.col("type")).alias("store_type"),
        F.col("cluster").cast("int").alias("cluster")
    )
    .dropDuplicates()
)

# Cada tienda debe tener un solo registro dimensional
duplicate_stores = (
    silver_stores
    .groupBy("store_nbr")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_stores.limit(1).count() > 0:
    display(duplicate_stores)
    raise ValueError(
        "Hay store_nbr duplicados en la dimensión de tiendas. "
        "Revisa los datos antes de hacer el join."
    )

display(silver_stores.limit(10))

In [0]:
silver_tables = {
    "silver_sales": silver_sales,
    "silver_transactions": silver_transactions,
    "silver_stores": silver_stores
}

for table_name, df in silver_tables.items():
    target = f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{table_name}`"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target)
    )

    print(f"Persistida: {CATALOG}.{SILVER_SCHEMA}.{table_name}")

In [0]:
sales = spark.table(
    f"`{CATALOG}`.`{SILVER_SCHEMA}`.`silver_sales`"
)

transactions = spark.table(
    f"`{CATALOG}`.`{SILVER_SCHEMA}`.`silver_transactions`"
).dropDuplicates()

stores = spark.table(
    f"`{CATALOG}`.`{SILVER_SCHEMA}`.`silver_stores`"
)

# Evitar duplicar ventas si hay más de una fila
# por fecha y tienda en transacciones.
duplicate_transactions = (
    transactions
    .groupBy("date", "store_nbr")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_transactions.limit(1).count() > 0:
    display(duplicate_transactions)
    raise ValueError(
        "Hay transacciones duplicadas por fecha y tienda. "
        "No se realizará el join hasta revisar esos duplicados."
    )

sales_enriched = (
    sales.alias("s")
    .join(
        stores.alias("st"),
        on="store_nbr",
        how="left"
    )
    .join(
        transactions.alias("t"),
        on=["date", "store_nbr"],
        how="left"
    )
    .select(
        "sales_id",
        "date",
        "store_nbr",
        "family",
        "sales",
        "onpromotion",
        "city",
        "state",
        "store_type",
        "cluster",
        "transactions"
    )
)

(
    sales_enriched.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"`{CATALOG}`.`{SILVER_SCHEMA}`.`silver_sales_enriched`"
    )
)

print("Tabla silver_sales_enriched creada.")

In [0]:
tables_to_check = [
    "silver_sales",
    "silver_transactions",
    "silver_stores",
    "silver_sales_enriched"
]

for table_name in tables_to_check:
    full_name = f"`{CATALOG}`.`{SILVER_SCHEMA}`.`{table_name}`"
    df = spark.table(full_name)

    detail = spark.sql(
        f"DESCRIBE DETAIL {full_name}"
    ).first()

    print(
        f"{table_name}: "
        f"{df.count()} filas | "
        f"{len(df.columns)} columnas | "
        f"{detail['format']}"
    )

    df.show(5, truncate=False)

In [0]:
sales_count = spark.table(
    "adbsflordeoro.retail_silver.silver_sales"
).count()

enriched_count = spark.table(
    "adbsflordeoro.retail_silver.silver_sales_enriched"
).count()

print("Filas de ventas:", sales_count)
print("Filas de ventas enriquecidas:", enriched_count)

if sales_count != enriched_count:
    raise ValueError(
        "El número de filas cambió después del JOIN. "
        "Revisa la unicidad de las claves."
    )

print("Validación de cardinalidad correcta.")

In [0]:
%sql
SELECT 'silver_sales' AS tabla, COUNT(*) AS registros
FROM adbsflordeoro.retail_silver.silver_sales

UNION ALL

SELECT 'silver_transactions', COUNT(*)
FROM adbsflordeoro.retail_silver.silver_transactions

UNION ALL

SELECT 'silver_stores', COUNT(*)
FROM adbsflordeoro.retail_silver.silver_stores

UNION ALL

SELECT 'silver_sales_enriched', COUNT(*)
FROM adbsflordeoro.retail_silver.silver_sales_enriched;

In [0]:
%sql
SELECT
    'BRONZE' AS capa,
    SUM(CAST(sales AS DOUBLE)) AS total_ventas
FROM adbsflordeoro.default.bronze_train

UNION ALL

SELECT
    'SILVER' AS capa,
    SUM(sales) AS total_ventas
FROM adbsflordeoro.retail_silver.silver_sales;

In [0]:
%sql
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT sales_id) AS ids_unicos,
    COUNT(*) - COUNT(DISTINCT sales_id) AS posibles_duplicados
FROM adbsflordeoro.retail_silver.silver_sales;

In [0]:
%sql
SELECT
    'BRONZE' AS capa,
    SUM(CAST(sales AS DOUBLE)) AS total_ventas
FROM adbsflordeoro.default.bronze_train

UNION ALL

SELECT
    'SILVER' AS capa,
    SUM(sales) AS total_ventas
FROM adbsflordeoro.retail_silver.silver_sales;